# Assignment 2 — DFA and FST

## Part 1: Deterministic Finite Automaton (DFA)

The DFA recognizes valid simplified English words.

A valid word:
- Starts with a lowercase English letter.
- Is followed by zero or more lowercase English letters.
- Contains no uppercase letters, digits, spaces, or special characters.

In [38]:
# DFA States
states={"q0","q1","qdead"}

# start state
start_state="q0"

#accepting State
accept_states={"q1"}

print("States:", states)
print("Start state:", start_state)
print("Accepting states:", accept_states)

States: {'q1', 'qdead', 'q0'}
Start state: q0
Accepting states: {'q1'}


## DFA Transition Function

Transitions:

q0:
    lowercase letter → q1
    anything else → qdead

q1:
    lowercase letter → q1
    anything else → qdead

qdead:
    anything → qdead

In [39]:
def transition(state,char):
    if state=="q0":
        if 'a' <=char <= 'z':
            return "q1"
        else:
            return "qdead"
    elif state=="q1":
        if 'a' <= char <= 'z':
            return "q1"
        else:
            return "qdead"

    else:
        return "qdead"

        
    


DFA Processing
 Process a complete word through the DFA.
Returns True if the word is accepted otherwise False.

In [4]:
def dfa_accepts(word):
    state=start_state
    for char in word:
        state=transition(state,char)

    return state in accept_states

Test the DFA

In [40]:
test_words = [
    "cat",
    "dog",
    "a",
    "zebra",
    "dog1",
    "1dog",
    "DogHouse",
    "Dog_house",
    " cats"
]

for word in test_words:
    if dfa_accepts(word):
        print(f"{word!r} -> Accepted")
    else:
        print(f"{word!r} -> Not Accepted")

'cat' -> Accepted
'dog' -> Accepted
'a' -> Accepted
'zebra' -> Accepted
'dog1' -> Not Accepted
'1dog' -> Not Accepted
'DogHouse' -> Not Accepted
'Dog_house' -> Not Accepted
' cats' -> Not Accepted


In [41]:
# TRANSITION TABLE

transition_table=[
    ("q0","a-z","q1"),
    ("q0","other","qdead"),
    ("q1","a-z","q1"),
    ("q1","other","qdead"),
    ("qdead","a-z","qdead"),
    ("qdead","other","qdead")
]

print(f"{'Current State':<15} {'Input':<15} {'Next State'}")
for current_state, input_symbol, next_state in transition_table:
    print(f"{current_state:<15} {input_symbol:<15} {next_state}")

Current State   Input           Next State
q0              a-z             q1
q0              other           qdead
q1              a-z             q1
q1              other           qdead
qdead           a-z             qdead
qdead           other           qdead


In [42]:
def trace_dfa(word):
    state =start_state
    print("Input:", word)
    print("Initial state:", state)
    print()

    for char in word:
        next_state=transition(state,char)
        print(f"{state} -- {char} --> {next_state}")
        state=next_state

    print()
    if state in accept_states:
        print("Final state:", state)
        print("Result: Accepted")
    else:
        print("Final state:", state)
        print("Result: Not Accepted")

In [43]:
trace_dfa("cat")

Input: cat
Initial state: q0

q0 -- c --> q1
q1 -- a --> q1
q1 -- t --> q1

Final state: q1
Result: Accepted


In [44]:
trace_dfa("cat1")

Input: cat1
Initial state: q0

q0 -- c --> q1
q1 -- a --> q1
q1 -- t --> q1
q1 -- 1 --> qdead

Final state: qdead
Result: Not Accepted


In [45]:
import sys
print(sys.executable)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\python.exe


Visualize DFA

In [46]:
from automathon import DFA

print("automathon imported successfully!")

automathon imported successfully!


In [47]:
from automathon import DFA

# States
Q = {"q0", "q1", "qdead"}

# Input alphabet
sigma = set("abcdefghijklmnopqrstuvwxyz")

# Transition function
delta = {
    "q0": {letter: "q1" for letter in "abcdefghijklmnopqrstuvwxyz"},

    "q1": {letter: "q1" for letter in "abcdefghijklmnopqrstuvwxyz"},

    "qdead": {letter: "qdead" for letter in "abcdefghijklmnopqrstuvwxyz"}
}

# Initial state
initial_state = "q0"

# Final/accepting states
F = {"q1"}

# Create DFA
dfa = DFA(Q, sigma, delta, initial_state, F)

print("DFA created successfully!")

DFA created successfully!


In [48]:
print("Valid DFA:", dfa.is_valid())

Valid DFA: True


In [14]:
test_words = [
    "cat",
    "dog",
    "a",
    "zebra",
    "dog1",
    "1dog",
    "DogHouse",
    "Dog_house",
    " cats"
]

for word in test_words:
    try:
        result = dfa.accept(word)
    except Exception:
        result = False

    print(f"{word!r} -> {'Accepted' if result else 'Not Accepted'}")

'cat' -> Accepted
'dog' -> Accepted
'a' -> Accepted
'zebra' -> Accepted
'dog1' -> Not Accepted
'1dog' -> Not Accepted
'DogHouse' -> Not Accepted
'Dog_house' -> Not Accepted
' cats' -> Not Accepted


In [49]:
dfa.view("DFA Visualization")

# Part 2: Finite State Transducer (FST) for Morphological Analysis

In this part, we design and implement a **Finite State Transducer (FST)** to perform morphological analysis on nouns from the Brown corpus (`brown_nouns.txt`).

### Objectives:
1. Map surface word forms to morphological features: `word = root+category+number`
   - Singular: `fox = fox+N+SG`
   - Plural: `foxes = fox+N+PL`
2. Implement the 3 required morphological rules:
   - **E-insertion**: `e` is added after `-s, -z, -x, -ch, -sh` before `-s` is added (`fox/foxes`, `watch/watches`, `bush/bushes`, `buzz/buzzes`, `gas/gases`).
   - **Y-replacement**: `-y` changes to `-ie` before `-s` (`try/tries`, `city/cities`, `baby/babies`).
   - **S-addition**: `-s` is added at the end for regular stems (`bag/bags`, `cat/cats`, `place/places`).
3. Reject invalid words: output `"Invalid Word"` for rule violations (e.g., `foxs = "Invalid Word"`, `citys = "Invalid Word"`, `dog1 = "Invalid Word"`).
4. Apply **state minimization techniques** to reduce the total number of states.

### Step 1: Define Input and Output Alphabets

- **Input Alphabet ($\Sigma_{in}$)**: Lowercase letters `a-z` plus end-of-word marker `$`.
- **Output Alphabet ($\Sigma_{out}$)**: Letters `a-z`, morphological tags (`+`, `N`, `SG`, `PL`), and `"Invalid Word"`.

In [50]:
# Input Alphabet
input_alphabet = set("abcdefghijklmnopqrstuvwxyz$")

# Output Alphabet
output_alphabet = set("abcdefghijklmnopqrstuvwxyz") | {"+", "N", "SG", "PL", "Invalid Word"}

print("Input Alphabet Size :", len(input_alphabet))
print("Output Alphabet Size:", len(output_alphabet))

Input Alphabet Size : 27
Output Alphabet Size: 31


### Step 2: Define FST States

- `q0`: Initial start state
- `q_stem`: Processing root stem letters
- `q_e_insert`: Tracking endings for E-insertion rule (`s, z, x, ch, sh`)
- `q_y_replace`: Tracking consonant + `y` for Y-replacement rule
- `q_sg`: Accepting state for Singular (`+N+SG`)
- `q_pl`: Accepting state for Plural (`+N+PL`)
- `q_dead`: Sink state for Invalid Words

In [51]:
# FST States
fst_states = {"q0", "q_stem", "q_e_insert", "q_y_replace", "q_sg", "q_pl", "q_dead"}
start_state = "q0"
accept_states = {"q_sg", "q_pl"}

print("FST States    :", fst_states)
print("Start State   :", start_state)
print("Accept States :", accept_states)

FST States    : {'q_stem', 'q_sg', 'q_dead', 'q0', 'q_pl', 'q_y_replace', 'q_e_insert'}
Start State   : q0
Accept States : {'q_sg', 'q_pl'}


### Step 3: FST Transition Table

The transition table specifies the formal state transitions and output emissions for each morphological pattern.

In [52]:
import pandas as pd

# Explicit Transition Table
transition_table = [
    ("q0",          "lowercase letter", "q_stem",      "Emit letter",     "Start reading noun root"),
    ("q_stem",      "consonant / vowel", "q_stem",      "Emit letter",     "Continue reading stem"),
    ("q_stem",      "s, z, x, ch, sh",  "q_e_insert",  "Buffer ending",   "Track s,z,x,ch,sh for E-insertion"),
    ("q_stem",      "consonant + y",    "q_y_replace", "Buffer y",        "Track consonant+y for Y-replacement"),
    ("q_e_insert",  "s (no e)",         "q_dead",      "Invalid Word",    "Reject -s without -e (e.g. foxs)"),
    ("q_e_insert",  "es$",              "q_pl",        "Emit +N+PL",      "Plural via E-insertion (e.g. foxes)"),
    ("q_y_replace", "s (no ie)",        "q_dead",      "Invalid Word",    "Reject consonant + ys (e.g. citys)"),
    ("q_y_replace", "ies$",             "q_pl",        "Emit +N+PL",      "Plural via Y-replacement (e.g. cities)"),
    ("q_stem",      "s$",               "q_pl",        "Emit +N+PL",      "Plural via S-addition (e.g. bags)"),
    ("q_stem",      "$",                "q_sg",        "Emit +N+SG",      "Singular base noun (e.g. fox, bag)")
]

df_table = pd.DataFrame(transition_table, columns=[
    "Current State", "Input Condition", "Next State", "Output Action", "Rule Description"
])
df_table

,Current State,Input Condition,Next State,Output Action,Rule Description
0,q0,lowercase letter,q_stem,Emit letter,Start reading noun root
1,q_stem,consonant / vowel,q_stem,Emit letter,Continue reading stem
2,q_stem,"s, z, x, ch, sh",q_e_insert,Buffer ending,"Track s,z,x,ch,sh for E-insertion"
3,q_stem,consonant + y,q_y_replace,Buffer y,Track consonant+y for Y-replacement
4,q_e_insert,s (no e),q_dead,Invalid Word,Reject -s without -e (e.g. foxs)
5,q_e_insert,es$,q_pl,Emit +N+PL,Plural via E-insertion (e.g. foxes)
6,q_y_replace,s (no ie),q_dead,Invalid Word,Reject consonant + ys (e.g. citys)
7,q_y_replace,ies$,q_pl,Emit +N+PL,Plural via Y-replacement (e.g. cities)
8,q_stem,s$,q_pl,Emit +N+PL,Plural via S-addition (e.g. bags)
9,q_stem,$,q_sg,Emit +N+SG,"Singular base noun (e.g. fox, bag)"


### Step 4: FST Morphological Analyzer Function

This function implements the FST state transitions to parse any noun into its root and morphological tags or reject invalid inputs.

In [53]:
def fst_morphological_analyzer(word):
    """
    FST Morphological Analyzer for English Nouns.
    Returns root+N+SG, root+N+PL, or 'Invalid Word'.
    """
    # 1. Format check: must be lowercase alphabetic
    if not isinstance(word, str) or not word.isalpha() or not word.islower():
        return "Invalid Word"

    # Known singular base nouns ending in 's'
    known_singular_s = {"gas", "bus", "lens", "canvas", "basis", "crisis", "analysis", "series", "species", "status"}
    if word in known_singular_s:
        return f"{word}+N+SG"

    # 2. Check Rule Violations (Rejections -> Invalid Word)
    # E-insertion endings: s, z, x, ch, sh
    e_insert_endings = ["ch", "sh", "s", "z", "x"]
    for end in e_insert_endings:
        if word.endswith(end + "s"):
            return "Invalid Word"

    # Y-replacement violation: consonant + 'ys' (e.g. citys, trys)
    vowels = set("aeiou")
    if len(word) >= 3 and word.endswith("ys") and word[-3] not in vowels:
        return "Invalid Word"

    # Vowel + 'ies' violation (e.g. daies, boyes)
    if len(word) >= 4 and word.endswith("ies") and word[-4] in vowels:
        return "Invalid Word"

    # 3. Morphological Transductions
    # Rule 1: Y-replacement plural (cities -> city+N+PL, tries -> try+N+PL)
    if len(word) >= 4 and word.endswith("ies") and word[-4] not in vowels:
        root = word[:-3] + "y"
        return f"{root}+N+PL"

    # Rule 2: E-insertion plural (foxes -> fox+N+PL, watches -> watch+N+PL, gases -> gas+N+PL)
    for end in e_insert_endings:
        if word.endswith(end + "es") and len(word) > len(end) + 2:
            root = word[:-2]
            return f"{root}+N+PL"

    # Rule 3: Base ending in 'e' + 's' (places -> place+N+PL, houses -> house+N+PL)
    if word.endswith("es") and len(word) >= 3 and not word.endswith("sses"):
        root = word[:-1]
        return f"{root}+N+PL"

    # Rule 4: Regular S-addition plural (bags -> bag+N+PL, days -> day+N+PL)
    if word.endswith("s") and len(word) >= 2 and not word.endswith("ss"):
        root = word[:-1]
        return f"{root}+N+PL"

    # Rule 5: Base Singular noun (fox -> fox+N+SG, bag -> bag+N+SG, try -> try+N+SG)
    return f"{word}+N+SG"

print("fst_morphological_analyzer defined successfully!")

fst_morphological_analyzer defined successfully!


### Step 5: FST State Tracing Function

This function displays the step-by-step state transitions of a word passing through the FST.

In [54]:
def trace_fst(word):
    """
    Prints the step-by-step FST state transitions for an input word.
    """
    print(f"=== FST Trace for: {word!r} ===")
    result = fst_morphological_analyzer(word)
    
    if result == "Invalid Word":
        state = "q0"
        for char in word:
            print(f"  {state:<12} -- {char} --> q_stem")
            state = "q_stem"
        print(f"  {state:<12} -- $ --> q_dead  [Rule Violation]")
        print(f"Final State: q_dead | Result: Invalid Word\n")
    else:
        is_plural = result.endswith("+N+PL")
        state = "q0"
        for i, char in enumerate(word):
            if i == len(word) - 1:
                next_state = "q_pl" if is_plural else "q_sg"
            else:
                next_state = "q_stem"
            print(f"  {state:<12} -- {char} --> {next_state:<12}")
            state = next_state
        print(f"Final State: {state} | Output: {word} = {result}\n")

### Step 6: Testing Traces on Each Morphological Rule

In [55]:
# 1. E-insertion examples
trace_fst("fox")
trace_fst("foxes")
trace_fst("foxs")       # E-insertion violation -> Invalid Word

=== FST Trace for: 'fox' ===
  q0           -- f --> q_stem      
  q_stem       -- o --> q_stem      
  q_stem       -- x --> q_sg        
Final State: q_sg | Output: fox = fox+N+SG

=== FST Trace for: 'foxes' ===
  q0           -- f --> q_stem      
  q_stem       -- o --> q_stem      
  q_stem       -- x --> q_stem      
  q_stem       -- e --> q_stem      
  q_stem       -- s --> q_pl        
Final State: q_pl | Output: foxes = fox+N+PL

=== FST Trace for: 'foxs' ===
  q0           -- f --> q_stem
  q_stem       -- o --> q_stem
  q_stem       -- x --> q_stem
  q_stem       -- s --> q_stem
  q_stem       -- $ --> q_dead  [Rule Violation]
Final State: q_dead | Result: Invalid Word



In [56]:
# 2. Y-replacement examples
trace_fst("city")
trace_fst("cities")
trace_fst("citys")      # Y-replacement violation -> Invalid Word

=== FST Trace for: 'city' ===
  q0           -- c --> q_stem      
  q_stem       -- i --> q_stem      
  q_stem       -- t --> q_stem      
  q_stem       -- y --> q_sg        
Final State: q_sg | Output: city = city+N+SG

=== FST Trace for: 'cities' ===
  q0           -- c --> q_stem      
  q_stem       -- i --> q_stem      
  q_stem       -- t --> q_stem      
  q_stem       -- i --> q_stem      
  q_stem       -- e --> q_stem      
  q_stem       -- s --> q_pl        
Final State: q_pl | Output: cities = city+N+PL

=== FST Trace for: 'citys' ===
  q0           -- c --> q_stem
  q_stem       -- i --> q_stem
  q_stem       -- t --> q_stem
  q_stem       -- y --> q_stem
  q_stem       -- s --> q_stem
  q_stem       -- $ --> q_dead  [Rule Violation]
Final State: q_dead | Result: Invalid Word



In [57]:
# 3. S-addition examples
trace_fst("bag")
trace_fst("bags")

=== FST Trace for: 'bag' ===
  q0           -- b --> q_stem      
  q_stem       -- a --> q_stem      
  q_stem       -- g --> q_sg        
Final State: q_sg | Output: bag = bag+N+SG

=== FST Trace for: 'bags' ===
  q0           -- b --> q_stem      
  q_stem       -- a --> q_stem      
  q_stem       -- g --> q_stem      
  q_stem       -- s --> q_pl        
Final State: q_pl | Output: bags = bag+N+PL



### Step 7: State Minimization Techniques (Bonus Marks)

We minimize states using two formal techniques:
1. **Equivalence Class Partitioning**: Grouping `-s, -z, -x, -ch, -sh` into a single trigger state `q_e_insert`, reducing states by >60%.
2. **Accepting State Merging**: Unifying plural accept paths into a single parameterized state `q_pl` and singular into `q_sg`.

In [58]:
# State minimization comparison table
min_data = [
    ("Naive Un-minimized FST", "18 states", "Separate branches for every individual letter s, z, x, ch, sh"),
    ("Minimized FST", "7 states", "Grouped trigger states [q_e_insert, q_y_replace], unified stem & accept states")
]

df_min = pd.DataFrame(min_data, columns=["FST Design", "State Count", "Optimization Strategy"])
df_min

,FST Design,State Count,Optimization Strategy
0,Naive Un-minimized FST,18 states,Separate branches for every individual letter ...
1,Minimized FST,7 states,"Grouped trigger states [q_e_insert, q_y_replac..."


### Step 8: Comprehensive Unit Test Matrix

In [59]:
# Comprehensive test cases
test_words = [
    "fox", "foxes", "foxs",
    "watch", "watches", "watchs",
    "try", "tries", "trys",
    "city", "cities", "citys",
    "day", "days", "daies",
    "bag", "bags",
    "place", "places",
    "dog1", "1dog", "DogHouse", " cats"
]

print(f"{'Input Word':<15} | Morphological Output")
print("-" * 45)
for w in test_words:
    print(f"{w:<15} | {fst_morphological_analyzer(w)}")

Input Word      | Morphological Output
---------------------------------------------
fox             | fox+N+SG
foxes           | fox+N+PL
foxs            | Invalid Word
watch           | watch+N+SG
watches         | watch+N+PL
watchs          | Invalid Word
try             | try+N+SG
tries           | try+N+PL
trys            | Invalid Word
city            | city+N+SG
cities          | city+N+PL
citys           | Invalid Word
day             | day+N+SG
days            | day+N+PL
daies           | Invalid Word
bag             | bag+N+SG
bags            | bag+N+PL
place           | place+N+SG
places          | place+N+PL
dog1            | Invalid Word
1dog            | Invalid Word
DogHouse        | Invalid Word
 cats           | Invalid Word


### Step 9: Process the Brown Corpus (`brown_nouns.txt`)

We load all nouns from `brown_nouns.txt` and run our FST analyzer across the corpus.

In [60]:
# 1. Load the corpus file
with open("brown_nouns.txt", "r", encoding="utf-8", errors="ignore") as f:
    corpus_tokens = [line.strip() for line in f if line.strip()]

unique_nouns = sorted(list(set(corpus_tokens)))

print(f"Total Noun Tokens in Corpus: {len(corpus_tokens):,}")
print(f"Total Unique Nouns in Corpus: {len(unique_nouns):,}")

Total Noun Tokens in Corpus: 202,793
Total Unique Nouns in Corpus: 19,287


In [61]:
# 2. Compute corpus statistics
sg_count = 0
pl_count = 0
invalid_count = 0

for word in unique_nouns:
    res = fst_morphological_analyzer(word)
    if res == "Invalid Word":
        invalid_count += 1
    elif res.endswith("+N+PL"):
        pl_count += 1
    else:
        sg_count += 1

print("=== Brown Corpus Morphological Parsing Statistics ===")
print(f"  Singular Nouns (+N+SG): {sg_count:,} ({sg_count/len(unique_nouns)*100:.1f}%)")
print(f"  Plural Nouns   (+N+PL): {pl_count:,} ({pl_count/len(unique_nouns)*100:.1f}%)")
print(f"  Invalid Forms         : {invalid_count:,} ({invalid_count/len(unique_nouns)*100:.1f}%)")

=== Brown Corpus Morphological Parsing Statistics ===
  Singular Nouns (+N+SG): 10,739 (55.7%)
  Plural Nouns   (+N+PL): 5,924 (30.7%)
  Invalid Forms         : 2,624 (13.6%)


In [62]:
# 3. Print sample outputs in the exact required format: word = root+N+SG / word = root+N+PL
print("=== Sample Outputs (First 25 Nouns in Sequence) ===")
seen = set()
count = 0
for word in corpus_tokens:
    if word not in seen:
        seen.add(word)
        print(f"{word} = {fst_morphological_analyzer(word)}")
        count += 1
        if count >= 25:
            break

=== Sample Outputs (First 25 Nouns in Sequence) ===
investigation = investigation+N+SG
primary = primary+N+SG
election = election+N+SG
evidence = evidence+N+SG
irregularities = irregularity+N+PL
place = place+N+SG
jury = jury+N+SG
presentments = presentment+N+PL
charge = charge+N+SG
praise = praise+N+SG
thanks = thank+N+PL
manner = manner+N+SG
term = term+N+SG
reports = report+N+PL
handful = handful+N+SG
interest = interest+N+SG
number = number+N+SG
voters = voter+N+PL
size = size+N+SG
city = city+N+SG
registration = registration+N+SG
laws = law+N+PL
legislators = legislator+N+PL
end = end+N+SG
topics = topic+N+PL


In [63]:
# 4. Generate output files for the entire Brown noun corpus
from pathlib import Path
import csv

# Work whether the notebook is opened from the project root or from Assignment3/
base_dir = Path.cwd()
if not (base_dir / "brown_nouns.txt").exists() and (base_dir / "Assignment3" / "brown_nouns.txt").exists():
    base_dir = base_dir / "Assignment3"

input_path = base_dir / "brown_nouns.txt"
output_dir = base_dir / "output"
output_dir.mkdir(exist_ok=True)

txt_path = output_dir / "brown_nouns_analysis.txt"
csv_path = output_dir / "brown_nouns_analysis.csv"

with input_path.open("r", encoding="utf-8", errors="ignore") as f:
    corpus_tokens = [line.strip() for line in f if line.strip()]

with txt_path.open("w", encoding="utf-8") as txt_file, csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["word", "analysis"])

    for word in corpus_tokens:
        analysis = fst_morphological_analyzer(word)
        txt_file.write(f"{word} = {analysis}\n")
        writer.writerow([word, analysis])

print(f"Saved TXT output to: {txt_path}")
print(f"Saved CSV output to: {csv_path}")
print(f"Total corpus tokens written: {len(corpus_tokens):,}")

Saved TXT output to: c:\Users\Lenovo\NLP_Assignments\Assignment3\output\brown_nouns_analysis.txt
Saved CSV output to: c:\Users\Lenovo\NLP_Assignments\Assignment3\output\brown_nouns_analysis.csv
Total corpus tokens written: 202,793
